# Experiment 09 — Compact UNSW-NB15 external validation

This notebook performs the **pre-specified, supplementary external validation** of the
accepted NSL-KDD study. It is intentionally narrow:

- Dataset: official UNSW-NB15 training/testing partitions.
- Task: binary Normal vs Attack.
- Target seed: 42 only.
- Conditions: non-private, DP-SGD ε≈4, and DP-SGD ε≈2.
- Architecture/training: unchanged 64–32 ReLU MLP, Adam, 30 epochs, batch 256.
- IDS threshold: F2 threshold selected on target-validation only.
- MIA: fixed score-only logistic regression and fixed label-aware loss threshold.
- Shadows: seeds 101, 202, 303, 404 train the score-only attacker; seed 505
  calibrates both attack operating thresholds.
- Official UNSW-NB15 test data: final IDS utility only.

This is **not** a new privacy-budget sweep, model search, or multi-seed replication.
Results must be reported as single-seed external evidence. ε≈4 is described only as the
utility-favoured tested DP setting from NSL-KDD; it is not called optimal.


## Before running

1. In Colab, choose **Runtime → Change runtime type → GPU**.
2. Put these two official files in
   `/content/drive/MyDrive/ML-DP-NID/data/unsw_nb15/`:
   - `UNSW_NB15_training-set.csv`
   - `UNSW_NB15_testing-set.csv`
3. Run all cells in order. Runtime caches are signature-namespaced and may be resumed.
4. A successful run ends with `EXPERIMENT 09: COMPLETE` and creates
   `/content/drive/MyDrive/ML-DP-NID/results/unsw_nb15_external_validation/experiment09_evidence.zip`.

The raw dataset is not copied into the evidence ZIP. Its exact SHA-256 hashes, sizes,
schema, and row counts are recorded. If the files are missing, the data-location cell
stops before any training and prints the exact expected paths.


## 1. Install and import dependencies


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

installed_opacus = (
    importlib.metadata.version("opacus")
    if importlib.util.find_spec("opacus") is not None
    else None
)
if installed_opacus != "1.6.0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "opacus==1.6.0"
    ])

print("Opacus 1.6.0 is available.")


In [ ]:
from __future__ import annotations

import copy
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import time
import warnings
import zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 120)
plt.style.use("seaborn-v0_8-whitegrid")

print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "opacus": __import__("opacus").__version__,
    "sklearn": sklearn.__version__,
    "device_available": "cuda" if torch.cuda.is_available() else "cpu",
})


## 2. Frozen protocol


In [ ]:
PROTOCOL_VERSION = "experiment09_unsw_nb15_external_validation_v1"
IMPLEMENTATION_REVISION = "2026-09-14a"
TARGET_SEED = 42
SHADOW_SEEDS = [101, 202, 303, 404, 505]
ATTACK_TRAIN_SHADOW_SEEDS = [101, 202, 303, 404]
CALIBRATION_SHADOW_SEED = 505

HIDDEN_DIMS = (64, 32)
EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

TARGET_EPSILONS = [4.0, 2.0]
MAX_GRAD_NORM = 1.0
ACCOUNTANT = "prv"
POISSON_SAMPLING = True
SECURE_MODE = False

BOOTSTRAP_N = 1000
MIN_FAMILY_MEMBERS = 30
MIN_FAMILY_NONMEMBERS = 30
RESUME = True

CONDITIONS = [
    {"condition": "non_private", "formal_dp": False, "target_epsilon": None},
    {"condition": "dp_eps_4", "formal_dp": True, "target_epsilon": 4.0},
    {"condition": "dp_eps_2", "formal_dp": True, "target_epsilon": 2.0},
]

FIXED_ATTACKS = [
    {
        "threat_model": "score_only_black_box",
        "attack_model": "logistic_regression",
        "feature_set": "prob_attack",
    },
    {
        "threat_model": "label_aware_audit",
        "attack_model": "loss_threshold",
        "feature_set": "loss",
    },
]

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

def stable_seed(*parts: object) -> int:
    value = "::".join(str(part) for part in parts)
    return int(hashlib.sha256(value.encode("utf-8")).hexdigest()[:8], 16)

set_all_seeds(TARGET_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

display(pd.DataFrame(CONDITIONS))
print("Device:", DEVICE)
print("Planned trainings: 3 targets + 15 condition-matched shadows = 18")


## 3. Resolve Drive paths and verify the official dataset files


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/ML-DP-NID")
configured_project_dir = os.environ.get("ML_DP_NID_DIR")
if configured_project_dir:
    PROJECT_DIR = Path(configured_project_dir)
elif Path("/content/drive/MyDrive").exists():
    PROJECT_DIR = DEFAULT_PROJECT_DIR
else:
    PROJECT_DIR = Path.cwd()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_DIR / "data" / "unsw_nb15"
RESULTS_DIR = PROJECT_DIR / "results" / "unsw_nb15_external_validation"
MODEL_DIR = PROJECT_DIR / "artifacts" / "models" / "unsw_nb15_external_validation"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def locate_dataset_file(filename: str) -> Path:
    candidates = [
        DATA_DIR / filename,
        PROJECT_DIR / filename,
        Path("/content") / filename,
        Path.cwd() / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    expected = DATA_DIR / filename
    raise FileNotFoundError(
        f"Missing {filename}. Download the official UNSW-NB15 training/testing "
        f"CSV partitions and place this file at:\n{expected}\n"
        "No training has started."
    )

TRAIN_FILE = locate_dataset_file("UNSW_NB15_training-set.csv")
TEST_FILE = locate_dataset_file("UNSW_NB15_testing-set.csv")

def calculate_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(json_safe(payload), file, indent=2, allow_nan=False, sort_keys=True)

def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

TRAIN_SHA256 = calculate_sha256(TRAIN_FILE)
TEST_SHA256 = calculate_sha256(TEST_FILE)

print({
    "project_dir": str(PROJECT_DIR),
    "train_file": str(TRAIN_FILE),
    "train_sha256": TRAIN_SHA256,
    "test_file": str(TEST_FILE),
    "test_sha256": TEST_SHA256,
})


## 4. Load and validate UNSW-NB15


In [ ]:
REQUIRED_COLUMNS = {
    "id", "dur", "proto", "service", "state", "spkts", "dpkts",
    "sbytes", "dbytes", "rate", "sttl", "dttl", "sload", "dload",
    "sloss", "dloss", "sinpkt", "dinpkt", "sjit", "djit", "swin",
    "stcpb", "dtcpb", "dwin", "tcprtt", "synack", "ackdat", "smean",
    "dmean", "trans_depth", "response_body_len", "ct_srv_src",
    "ct_state_ttl", "ct_dst_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
    "ct_dst_src_ltm", "is_ftp_login", "ct_ftp_cmd", "ct_flw_http_mthd",
    "ct_src_ltm", "ct_srv_dst", "is_sm_ips_ports", "attack_cat", "label",
}
CATEGORICAL_FEATURES = ["proto", "service", "state"]

def clean_unsw(dataframe: pd.DataFrame, source: str) -> pd.DataFrame:
    dataframe = dataframe.copy()
    dataframe.columns = [
        str(column).strip().lower().replace(" ", "_")
        for column in dataframe.columns
    ]
    dataframe = dataframe.loc[:, ~dataframe.columns.str.startswith("unnamed")]
    missing = sorted(REQUIRED_COLUMNS - set(dataframe.columns))
    assert not missing, f"{source} is missing required columns: {missing}"

    dataframe = dataframe[list(sorted(REQUIRED_COLUMNS))].copy()
    dataframe["label"] = pd.to_numeric(dataframe["label"], errors="raise").astype(int)
    assert set(dataframe["label"].unique()) == {0, 1}

    attack_cat = dataframe["attack_cat"].fillna("").astype(str).str.strip()
    attack_cat = attack_cat.mask(
        (dataframe["label"] == 0) | attack_cat.eq(""),
        "Normal",
    )
    dataframe["attack_family"] = attack_cat
    dataframe["binary_label"] = dataframe["label"].astype(int)
    dataframe["source_row"] = np.arange(len(dataframe), dtype=np.int64)
    dataframe["record_id"] = [f"{source}:{index}" for index in dataframe["source_row"]]

    for column in CATEGORICAL_FEATURES:
        dataframe[column] = (
            dataframe[column].fillna("__MISSING__").astype(str).str.strip()
        )
        dataframe[column] = dataframe[column].replace("", "__MISSING__")

    return dataframe

train_df = clean_unsw(pd.read_csv(TRAIN_FILE, low_memory=False), "official_train")
test_df = clean_unsw(pd.read_csv(TEST_FILE, low_memory=False), "official_test")

assert len(train_df) == 175_341, (
    "Expected the official 175,341-row UNSW-NB15 training partition; "
    f"found {len(train_df):,}."
)
assert len(test_df) == 82_332, (
    "Expected the official 82,332-row UNSW-NB15 testing partition; "
    f"found {len(test_df):,}."
)

FEATURES = sorted(REQUIRED_COLUMNS - {"id", "attack_cat", "label"})
NUMERIC_FEATURES = [
    column for column in FEATURES if column not in CATEGORICAL_FEATURES
]

for dataframe, source in [(train_df, "official_train"), (test_df, "official_test")]:
    for column in NUMERIC_FEATURES:
        dataframe[column] = pd.to_numeric(dataframe[column], errors="coerce")
    dataframe[NUMERIC_FEATURES] = dataframe[NUMERIC_FEATURES].replace(
        [np.inf, -np.inf], np.nan
    )
    all_missing = [
        column for column in NUMERIC_FEATURES
        if dataframe[column].isna().all()
    ]
    assert not all_missing, f"All values are missing in {source}: {all_missing}"

dataset_identity = {
    "dataset": "UNSW-NB15",
    "source": "official predefined training/testing CSV partitions",
    "official_information_page": (
        "https://research.unsw.edu.au/projects/unsw-nb15-dataset"
    ),
    "train_filename": TRAIN_FILE.name,
    "test_filename": TEST_FILE.name,
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "train_sha256": TRAIN_SHA256,
    "test_sha256": TEST_SHA256,
    "raw_columns": sorted(REQUIRED_COLUMNS),
    "model_features": FEATURES,
    "excluded_columns": ["id", "attack_cat", "label"],
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "task": "binary Normal versus Attack",
}
write_json(RESULTS_DIR / "unsw_nb15_dataset_identity.json", dataset_identity)

display(pd.DataFrame([
    {
        "partition": "official_train",
        "rows": len(train_df),
        "normal": int((train_df["binary_label"] == 0).sum()),
        "attack": int((train_df["binary_label"] == 1).sum()),
        "families": int(train_df["attack_family"].nunique()),
    },
    {
        "partition": "official_test",
        "rows": len(test_df),
        "normal": int((test_df["binary_label"] == 0).sum()),
        "attack": int((test_df["binary_label"] == 1).sum()),
        "families": int(test_df["attack_family"].nunique()),
    },
]))


## 5. Create the locked 70/10/20 development split


In [ ]:
protocol_core = {
    "protocol_version": PROTOCOL_VERSION,
    "implementation_revision": IMPLEMENTATION_REVISION,
    "dataset_hashes": {"train": TRAIN_SHA256, "test": TEST_SHA256},
    "target_seed": TARGET_SEED,
    "shadow_seeds": SHADOW_SEEDS,
    "conditions": CONDITIONS,
    "fixed_attacks": FIXED_ATTACKS,
    "development_split": {
        "target_train": 0.70,
        "target_validation": 0.10,
        "shadow_pool": 0.20,
        "stratification": "binary_label + attack_family",
    },
    "features": {
        "included": FEATURES,
        "excluded": ["id", "attack_cat", "label"],
        "categorical": CATEGORICAL_FEATURES,
    },
    "architecture": {"hidden_dims": list(HIDDEN_DIMS), "activation": "ReLU"},
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "optimizer": "Adam",
    },
    "dp": {
        "max_grad_norm": MAX_GRAD_NORM,
        "accountant": ACCOUNTANT,
        "poisson_sampling": POISSON_SAMPLING,
        "secure_mode": SECURE_MODE,
    },
}
PROTOCOL_FINGERPRINT = hashlib.sha256(
    json.dumps(protocol_core, sort_keys=True).encode("utf-8")
).hexdigest()
RUN_DIR = RESULTS_DIR / "runtime" / PROTOCOL_FINGERPRINT[:16]
RUN_DIR.mkdir(parents=True, exist_ok=True)

split_key = (
    train_df["binary_label"].astype(str)
    + "__"
    + train_df["attack_family"].astype(str)
)
all_indices = np.arange(len(train_df), dtype=np.int64)
target_development_indices, shadow_pool_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=TARGET_SEED,
    stratify=split_key,
)
target_train_indices, target_validation_indices = train_test_split(
    target_development_indices,
    test_size=0.125,
    random_state=TARGET_SEED,
    stratify=split_key.iloc[target_development_indices],
)

combined_indices = np.concatenate([
    target_train_indices,
    target_validation_indices,
    shadow_pool_indices,
])
assert len(combined_indices) == len(train_df)
assert len(np.unique(combined_indices)) == len(train_df)
assert set(combined_indices) == set(all_indices)

target_train = train_df.iloc[target_train_indices].copy().reset_index(drop=True)
target_validation = train_df.iloc[target_validation_indices].copy().reset_index(drop=True)
shadow_pool = train_df.iloc[shadow_pool_indices].copy().reset_index(drop=True)

split_indices_file = RUN_DIR / "unsw_nb15_split_indices.npz"
np.savez_compressed(
    split_indices_file,
    target_train_indices=target_train_indices,
    target_validation_indices=target_validation_indices,
    shadow_pool_indices=shadow_pool_indices,
)

def split_row(name: str, dataframe: pd.DataFrame) -> dict:
    return {
        "partition": name,
        "rows": len(dataframe),
        "fraction_of_official_train": len(dataframe) / len(train_df),
        "normal": int((dataframe["binary_label"] == 0).sum()),
        "attack": int((dataframe["binary_label"] == 1).sum()),
        "attack_families": int(dataframe["attack_family"].nunique()),
    }

split_manifest = pd.DataFrame([
    split_row("target_train", target_train),
    split_row("target_validation", target_validation),
    split_row("shadow_pool", shadow_pool),
    split_row("official_test_final_utility_only", test_df),
])
split_manifest.to_csv(RESULTS_DIR / "unsw_nb15_split_manifest.csv", index=False)

display(split_manifest)
print("Protocol fingerprint:", PROTOCOL_FINGERPRINT)
print("Runtime cache namespace:", RUN_DIR)


## 6. Fit target preprocessing on target-train only


In [ ]:
def create_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def create_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", MinMaxScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("one_hot", create_one_hot_encoder()),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ])

def to_float32_dense(values) -> np.ndarray:
    if hasattr(values, "toarray"):
        values = values.toarray()
    values = np.asarray(values, dtype=np.float32)
    assert np.isfinite(values).all()
    return values

target_preprocessor = create_preprocessor()
X_target_train = to_float32_dense(
    target_preprocessor.fit_transform(target_train[FEATURES])
)
X_target_validation = to_float32_dense(
    target_preprocessor.transform(target_validation[FEATURES])
)
X_test = to_float32_dense(target_preprocessor.transform(test_df[FEATURES]))

y_target_train = target_train["binary_label"].to_numpy(
    dtype=np.float32
).reshape(-1, 1)
y_target_validation = target_validation["binary_label"].to_numpy(
    dtype=np.float32
).reshape(-1, 1)
y_test = test_df["binary_label"].to_numpy(dtype=np.float32).reshape(-1, 1)

TARGET_INPUT_DIM = X_target_train.shape[1]
TARGET_DELTA = 1.0 / len(X_target_train)

preprocessor_file = RUN_DIR / "target_preprocessor.joblib"
joblib.dump(target_preprocessor, preprocessor_file)
PREPROCESSOR_SHA256 = calculate_sha256(preprocessor_file)

preprocessing_manifest = {
    "fit_partition": "target_train_only",
    "fit_rows": len(target_train),
    "numeric": "median imputation then min-max scaling",
    "categorical": "most-frequent imputation then one-hot; unknown ignored",
    "excluded_label_metadata": ["id", "attack_cat", "label"],
    "input_feature_count": len(FEATURES),
    "transformed_feature_count": TARGET_INPUT_DIM,
    "preprocessor_sha256": PREPROCESSOR_SHA256,
    "dp_scope": "DP-SGD optimisation conditional on this fixed preprocessing",
}
write_json(RESULTS_DIR / "unsw_nb15_preprocessing_manifest.json", preprocessing_manifest)

print({
    "raw_feature_count": len(FEATURES),
    "transformed_feature_count": TARGET_INPUT_DIM,
    "target_delta": TARGET_DELTA,
    "preprocessor_sha256": PREPROCESSOR_SHA256,
})


## 7. Freeze the target MIA evaluation records


In [ ]:
def balanced_member_nonmember_sample(
    member_candidates: pd.DataFrame,
    nonmember_candidates: pd.DataFrame,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    members = []
    nonmembers = []
    common_families = sorted(
        set(member_candidates["attack_family"])
        & set(nonmember_candidates["attack_family"])
    )
    for family in common_families:
        member_group = member_candidates[
            member_candidates["attack_family"] == family
        ]
        nonmember_group = nonmember_candidates[
            nonmember_candidates["attack_family"] == family
        ]
        sample_size = min(len(member_group), len(nonmember_group))
        if sample_size == 0:
            continue
        members.append(member_group.sample(sample_size, random_state=seed))
        nonmembers.append(
            nonmember_group.sample(sample_size, random_state=seed + 1)
        )
    assert members and nonmembers
    return pd.concat(members), pd.concat(nonmembers)

target_member_records, target_nonmember_records = balanced_member_nonmember_sample(
    target_train,
    target_validation,
    TARGET_SEED,
)
target_member_positions = target_member_records.index.to_numpy()
target_nonmember_positions = target_nonmember_records.index.to_numpy()

target_mia_sample_manifest = pd.concat([
    pd.DataFrame({
        "membership": 1,
        "partition": "target_train",
        "partition_position": target_member_positions,
        "record_id": target_member_records["record_id"].to_numpy(),
        "true_label": target_member_records["binary_label"].to_numpy(),
        "attack_family": target_member_records["attack_family"].to_numpy(),
    }),
    pd.DataFrame({
        "membership": 0,
        "partition": "target_validation",
        "partition_position": target_nonmember_positions,
        "record_id": target_nonmember_records["record_id"].to_numpy(),
        "true_label": target_nonmember_records["binary_label"].to_numpy(),
        "attack_family": target_nonmember_records["attack_family"].to_numpy(),
    }),
], ignore_index=True).sample(frac=1, random_state=TARGET_SEED).reset_index(drop=True)

assert target_mia_sample_manifest.groupby("attack_family")["membership"].sum().equals(
    target_mia_sample_manifest.groupby("attack_family")["membership"].apply(
        lambda values: int((values == 0).sum())
    )
)
target_mia_sample_manifest.to_csv(
    RESULTS_DIR / "unsw_nb15_target_mia_sample_manifest.csv",
    index=False,
)

display(pd.crosstab(
    target_mia_sample_manifest["attack_family"],
    target_mia_sample_manifest["membership"],
))


## 8. Shared model, training, IDS, and MIA helpers


In [ ]:
class BinaryMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, HIDDEN_DIMS[0]),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1]),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIMS[1], 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)

def create_initial_state(input_dim: int, seed: int) -> dict:
    set_all_seeds(seed)
    return copy.deepcopy(BinaryMLP(input_dim).state_dict())

def unwrap_model(model: nn.Module) -> nn.Module:
    return getattr(model, "_module", model)

def create_dataset(features: np.ndarray, labels: np.ndarray) -> TensorDataset:
    return TensorDataset(torch.from_numpy(features), torch.from_numpy(labels))

def create_train_loader(
    features: np.ndarray,
    labels: np.ndarray,
    seed: int,
) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        create_dataset(features, labels),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
        drop_last=False,
    )

def create_eval_loader(features: np.ndarray, labels: np.ndarray) -> DataLoader:
    return DataLoader(
        create_dataset(features, labels),
        batch_size=2048,
        shuffle=False,
        num_workers=0,
    )

def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
) -> float:
    model.train()
    weighted_loss = 0.0
    total = 0
    for features, labels in data_loader:
        features = features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(features), labels)
        loss.backward()
        optimizer.step()
        weighted_loss += float(loss.detach().cpu()) * labels.shape[0]
        total += labels.shape[0]
    return weighted_loss / max(total, 1)

@torch.no_grad()
def predict_probabilities(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
) -> np.ndarray:
    model.eval()
    batches = []
    for batch_features, _ in create_eval_loader(features, labels):
        batch_features = batch_features.to(DEVICE, non_blocking=True)
        batches.append(torch.sigmoid(model(batch_features)).cpu().numpy())
    return np.concatenate(batches).reshape(-1)

def summarize_warnings(caught, stage: str, condition: str, seed: int) -> list[dict]:
    counts = Counter(
        (warning.category.__name__, str(warning.message))
        for warning in caught
    )
    return [
        {
            "stage": stage,
            "condition": condition,
            "seed": seed,
            "category": category,
            "message": message,
            "occurrences": count,
        }
        for (category, message), count in counts.items()
    ]

def select_f2_threshold(y_true: np.ndarray, probabilities: np.ndarray):
    rows = []
    for threshold in np.arange(0.01, 1.00, 0.01):
        predictions = (probabilities >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            "f2": fbeta_score(y_true, predictions, beta=2, zero_division=0),
            "recall": recall_score(y_true, predictions, zero_division=0),
            "f1": f1_score(y_true, predictions, zero_division=0),
        })
    search = pd.DataFrame(rows)
    selected = float(
        search.sort_values(
            ["f2", "recall", "threshold"],
            ascending=[False, False, True],
        ).iloc[0]["threshold"]
    )
    return selected, search

def calculate_ids_metrics(
    condition: str,
    formal_dp: bool,
    actual_epsilon: float | None,
    split_name: str,
    threshold_policy: str,
    y_true: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "condition": condition,
        "formal_dp": formal_dp,
        "target_epsilon": next(
            item["target_epsilon"] for item in CONDITIONS
            if item["condition"] == condition
        ),
        "actual_epsilon": actual_epsilon,
        "delta": TARGET_DELTA if formal_dp else np.nan,
        "split": split_name,
        "threshold_policy": threshold_policy,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "fnr": fn / (fn + tp) if fn + tp else np.nan,
        "fpr": fp / (fp + tn) if fp + tn else np.nan,
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def create_mia_feature_frame(
    condition: str,
    source: str,
    seed: int,
    membership: int,
    dataframe: pd.DataFrame,
    probabilities: np.ndarray,
) -> pd.DataFrame:
    true_labels = dataframe["binary_label"].to_numpy(dtype=int)
    true_probability = np.where(true_labels == 1, probabilities, 1 - probabilities)
    true_probability = np.clip(true_probability, 1e-12, 1 - 1e-12)
    return pd.DataFrame({
        "condition": condition,
        "source": source,
        "seed": seed,
        "record_id": dataframe["record_id"].to_numpy(),
        "membership": membership,
        "true_label": true_labels,
        "attack_family": dataframe["attack_family"].to_numpy(),
        "prob_attack": probabilities,
        "confidence": np.maximum(probabilities, 1 - probabilities),
        "loss": -np.log(true_probability),
        "correctness": ((probabilities >= 0.5).astype(int) == true_labels).astype(int),
    })

assert not ModuleValidator.validate(BinaryMLP(TARGET_INPUT_DIM), strict=False)
TARGET_INITIAL_STATE = create_initial_state(TARGET_INPUT_DIM, TARGET_SEED)
print("Shared MLP is Opacus-compatible.")


## 9. Train or resume the three target conditions


In [ ]:
@dataclass
class TargetResult:
    condition: str
    formal_dp: bool
    target_epsilon: float | None
    actual_epsilon: float | None
    delta: float | None
    noise_multiplier: float | None
    max_grad_norm: float | None
    sample_rate: float | None
    selected_threshold: float
    model_path: Path
    ids: pd.DataFrame
    mia_features: pd.DataFrame
    history: pd.DataFrame
    threshold_search: pd.DataFrame
    warnings: list[dict]
    cache_signature: str

def target_cache_paths(condition: str) -> dict[str, Path]:
    directory = RUN_DIR / "targets" / condition
    directory.mkdir(parents=True, exist_ok=True)
    return {
        "model": directory / "model.pt",
        "config": directory / "config.json",
        "ids": directory / "ids.csv",
        "mia": directory / "mia_features.csv",
        "history": directory / "history.csv",
        "threshold": directory / "threshold_search.csv",
        "warnings": directory / "warnings.csv",
    }

def target_signature(spec: dict) -> str:
    payload = {
        "protocol_fingerprint": PROTOCOL_FINGERPRINT,
        "condition": spec,
        "target_delta": TARGET_DELTA,
        "preprocessor_sha256": PREPROCESSOR_SHA256,
        "target_train_record_ids": hashlib.sha256(
            "\n".join(target_train["record_id"]).encode("utf-8")
        ).hexdigest(),
    }
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode("utf-8")
    ).hexdigest()

def build_target_mia_features(
    condition: str,
    train_probabilities: np.ndarray,
    validation_probabilities: np.ndarray,
) -> pd.DataFrame:
    lookup = pd.concat([
        pd.DataFrame({
            "membership": 1,
            "partition_position": target_member_positions,
            "prob_attack": train_probabilities[target_member_positions],
        }),
        pd.DataFrame({
            "membership": 0,
            "partition_position": target_nonmember_positions,
            "prob_attack": validation_probabilities[target_nonmember_positions],
        }),
    ], ignore_index=True)
    merged = target_mia_sample_manifest.merge(
        lookup,
        on=["membership", "partition_position"],
        validate="one_to_one",
    )
    labels = merged["true_label"].to_numpy(dtype=int)
    probabilities = merged["prob_attack"].to_numpy()
    true_probability = np.where(labels == 1, probabilities, 1 - probabilities)
    true_probability = np.clip(true_probability, 1e-12, 1 - 1e-12)
    merged.insert(0, "condition", condition)
    merged["confidence"] = np.maximum(probabilities, 1 - probabilities)
    merged["loss"] = -np.log(true_probability)
    merged["correctness"] = (
        (probabilities >= 0.5).astype(int) == labels
    ).astype(int)
    return merged

def cache_is_valid(paths: dict[str, Path], signature: str) -> bool:
    required = ["model", "config", "ids", "mia", "history", "threshold"]
    if not all(paths[name].exists() for name in required):
        return False
    config = load_json(paths["config"])
    return bool(
        config.get("cache_signature") == signature
        and config.get("model_sha256") == calculate_sha256(paths["model"])
        and config.get("mia_features_sha256") == calculate_sha256(paths["mia"])
        and config.get("ids_sha256") == calculate_sha256(paths["ids"])
        and config.get("history_sha256") == calculate_sha256(paths["history"])
        and config.get("threshold_search_sha256") == calculate_sha256(paths["threshold"])
    )

def train_target(spec: dict) -> TargetResult:
    condition = spec["condition"]
    formal_dp = spec["formal_dp"]
    requested_epsilon = spec["target_epsilon"]
    signature = target_signature(spec)
    paths = target_cache_paths(condition)

    if RESUME and cache_is_valid(paths, signature):
        config = load_json(paths["config"])
        warnings_rows = (
            pd.read_csv(paths["warnings"]).to_dict(orient="records")
            if paths["warnings"].exists() else []
        )
        print(f"Resuming validated target cache: {condition}")
        return TargetResult(
            condition, formal_dp, requested_epsilon,
            config.get("actual_epsilon"), config.get("delta"),
            config.get("noise_multiplier"), config.get("max_grad_norm"),
            config.get("sample_rate"), config["selected_threshold"],
            paths["model"], pd.read_csv(paths["ids"]),
            pd.read_csv(paths["mia"]), pd.read_csv(paths["history"]),
            pd.read_csv(paths["threshold"]), warnings_rows, signature,
        )

    set_all_seeds(TARGET_SEED)
    model = BinaryMLP(TARGET_INPUT_DIM)
    model.load_state_dict(TARGET_INITIAL_STATE)
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    criterion = nn.BCEWithLogitsLoss()
    train_loader = create_train_loader(
        X_target_train, y_target_train, TARGET_SEED
    )
    privacy_engine = None
    noise_multiplier = None
    actual_max_grad_norm = None
    sample_rate = None
    warning_rows = []

    if formal_dp:
        privacy_engine = PrivacyEngine(accountant=ACCOUNTANT, secure_mode=SECURE_MODE)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("once")
            model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
                module=model,
                optimizer=optimizer,
                criterion=criterion,
                data_loader=train_loader,
                target_epsilon=requested_epsilon,
                target_delta=TARGET_DELTA,
                epochs=EPOCHS,
                max_grad_norm=MAX_GRAD_NORM,
                poisson_sampling=POISSON_SAMPLING,
                clipping="flat",
                loss_reduction="mean",
            )
        warning_rows.extend(
            summarize_warnings(caught, "target_make_private", condition, TARGET_SEED)
        )
        noise_multiplier = float(optimizer.noise_multiplier)
        actual_max_grad_norm = float(optimizer.max_grad_norm)
        sample_rate = float(
            getattr(train_loader, "sample_rate", 1.0 / len(train_loader))
        )

    history_rows = []
    started_at = time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("once")
        for epoch in range(1, EPOCHS + 1):
            loss = train_one_epoch(model, train_loader, optimizer, criterion)
            epsilon = (
                float(privacy_engine.get_epsilon(TARGET_DELTA))
                if formal_dp else np.nan
            )
            history_rows.append({
                "condition": condition,
                "seed": TARGET_SEED,
                "epoch": epoch,
                "train_loss": loss,
                "epsilon": epsilon,
                "delta": TARGET_DELTA if formal_dp else np.nan,
            })
            if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
                epsilon_text = f", epsilon={epsilon:.4f}" if formal_dp else ""
                print(
                    f"{condition}: epoch {epoch:02d}/{EPOCHS}, "
                    f"loss={loss:.6f}{epsilon_text}"
                )
    warning_rows.extend(
        summarize_warnings(caught, "target_training", condition, TARGET_SEED)
    )
    training_seconds = time.time() - started_at

    actual_epsilon = (
        float(privacy_engine.get_epsilon(TARGET_DELTA)) if formal_dp else None
    )
    if formal_dp:
        assert np.isfinite(actual_epsilon)
        assert 0 < actual_epsilon <= requested_epsilon + 0.10

    train_probabilities = predict_probabilities(
        model, X_target_train, y_target_train
    )
    validation_probabilities = predict_probabilities(
        model, X_target_validation, y_target_validation
    )
    test_probabilities = predict_probabilities(model, X_test, y_test)

    selected_threshold, threshold_search = select_f2_threshold(
        y_target_validation.reshape(-1).astype(int), validation_probabilities
    )
    ids = pd.DataFrame([
        calculate_ids_metrics(
            condition, formal_dp, actual_epsilon,
            "target_validation", "default_0_5",
            y_target_validation.reshape(-1).astype(int),
            validation_probabilities, 0.5,
        ),
        calculate_ids_metrics(
            condition, formal_dp, actual_epsilon,
            "official_UNSW_NB15_test", "default_0_5",
            y_test.reshape(-1).astype(int), test_probabilities, 0.5,
        ),
        calculate_ids_metrics(
            condition, formal_dp, actual_epsilon,
            "official_UNSW_NB15_test", "validation_selected_F2",
            y_test.reshape(-1).astype(int), test_probabilities,
            selected_threshold,
        ),
    ])
    mia_features = build_target_mia_features(
        condition, train_probabilities, validation_probabilities
    )
    history = pd.DataFrame(history_rows)

    torch.save(unwrap_model(model).state_dict(), paths["model"])
    ids.to_csv(paths["ids"], index=False)
    mia_features.to_csv(paths["mia"], index=False)
    history.to_csv(paths["history"], index=False)
    threshold_search.assign(condition=condition).to_csv(
        paths["threshold"], index=False
    )
    pd.DataFrame(
        warning_rows,
        columns=["stage", "condition", "seed", "category", "message", "occurrences"],
    ).to_csv(paths["warnings"], index=False)

    config = {
        "cache_signature": signature,
        "condition": condition,
        "formal_dp": formal_dp,
        "target_epsilon": requested_epsilon,
        "actual_epsilon": actual_epsilon,
        "delta": TARGET_DELTA if formal_dp else None,
        "noise_multiplier": noise_multiplier,
        "max_grad_norm": actual_max_grad_norm,
        "sample_rate": sample_rate,
        "selected_threshold": selected_threshold,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "training_seconds": training_seconds,
        "model_sha256": calculate_sha256(paths["model"]),
        "mia_features_sha256": calculate_sha256(paths["mia"]),
        "ids_sha256": calculate_sha256(paths["ids"]),
        "history_sha256": calculate_sha256(paths["history"]),
        "threshold_search_sha256": calculate_sha256(paths["threshold"]),
    }
    write_json(paths["config"], config)

    del model, optimizer, train_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return TargetResult(
        condition, formal_dp, requested_epsilon, actual_epsilon,
        TARGET_DELTA if formal_dp else None, noise_multiplier,
        actual_max_grad_norm, sample_rate, selected_threshold,
        paths["model"], ids, mia_features, history, threshold_search,
        warning_rows, signature,
    )

target_results = {spec["condition"]: train_target(spec) for spec in CONDITIONS}
print("All three target conditions are complete or resumed from validated caches.")


## 10. Train or resume 15 condition-matched shadow models


In [ ]:
def create_shadow_split(dataframe: pd.DataFrame, seed: int):
    key = (
        dataframe["binary_label"].astype(str)
        + "__"
        + dataframe["attack_family"].astype(str)
    )
    member_indices, nonmember_indices = train_test_split(
        np.arange(len(dataframe)),
        test_size=0.50,
        random_state=seed,
        stratify=key,
    )
    return (
        dataframe.iloc[member_indices].copy(),
        dataframe.iloc[nonmember_indices].copy(),
    )

def shadow_paths(condition: str, seed: int) -> dict[str, Path]:
    directory = RUN_DIR / "shadows" / condition / f"seed_{seed}"
    directory.mkdir(parents=True, exist_ok=True)
    return {
        "features": directory / "mia_features.csv",
        "config": directory / "config.json",
        "history": directory / "history.csv",
        "warnings": directory / "warnings.csv",
    }

def shadow_signature(spec: dict, seed: int) -> str:
    payload = {
        "protocol_fingerprint": PROTOCOL_FINGERPRINT,
        "condition": spec,
        "shadow_seed": seed,
        "target_delta": TARGET_DELTA,
        "shadow_pool_record_ids": hashlib.sha256(
            "\n".join(shadow_pool["record_id"]).encode("utf-8")
        ).hexdigest(),
    }
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode("utf-8")
    ).hexdigest()

def train_shadow(spec: dict, seed: int):
    condition = spec["condition"]
    formal_dp = spec["formal_dp"]
    requested_epsilon = spec["target_epsilon"]
    signature = shadow_signature(spec, seed)
    paths = shadow_paths(condition, seed)

    if RESUME and paths["features"].exists() and paths["config"].exists():
        config = load_json(paths["config"])
        valid = bool(
            config.get("cache_signature") == signature
            and config.get("features_sha256") == calculate_sha256(paths["features"])
            and paths["history"].exists()
            and config.get("history_sha256") == calculate_sha256(paths["history"])
        )
        if valid:
            warning_rows = (
                pd.read_csv(paths["warnings"]).to_dict(orient="records")
                if paths["warnings"].exists() else []
            )
            print(f"Resuming validated shadow cache: {condition}, seed {seed}")
            return pd.read_csv(paths["features"]), config, warning_rows

    shadow_members, shadow_nonmembers = create_shadow_split(shadow_pool, seed)
    preprocessor = create_preprocessor()
    X_members = to_float32_dense(
        preprocessor.fit_transform(shadow_members[FEATURES])
    )
    X_nonmembers = to_float32_dense(
        preprocessor.transform(shadow_nonmembers[FEATURES])
    )
    y_members = shadow_members["binary_label"].to_numpy(
        dtype=np.float32
    ).reshape(-1, 1)
    y_nonmembers = shadow_nonmembers["binary_label"].to_numpy(
        dtype=np.float32
    ).reshape(-1, 1)

    input_dim = X_members.shape[1]
    set_all_seeds(seed)
    model = BinaryMLP(input_dim)
    model.load_state_dict(create_initial_state(input_dim, seed))
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    criterion = nn.BCEWithLogitsLoss()
    train_loader = create_train_loader(X_members, y_members, seed)
    privacy_engine = None
    noise_multiplier = None
    sample_rate = None
    warning_rows = []

    if formal_dp:
        privacy_engine = PrivacyEngine(accountant=ACCOUNTANT, secure_mode=SECURE_MODE)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("once")
            model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
                module=model,
                optimizer=optimizer,
                criterion=criterion,
                data_loader=train_loader,
                target_epsilon=requested_epsilon,
                target_delta=TARGET_DELTA,
                epochs=EPOCHS,
                max_grad_norm=MAX_GRAD_NORM,
                poisson_sampling=POISSON_SAMPLING,
                clipping="flat",
                loss_reduction="mean",
            )
        warning_rows.extend(
            summarize_warnings(caught, "shadow_make_private", condition, seed)
        )
        noise_multiplier = float(optimizer.noise_multiplier)
        sample_rate = float(
            getattr(train_loader, "sample_rate", 1.0 / len(train_loader))
        )

    history_rows = []
    started_at = time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("once")
        for epoch in range(1, EPOCHS + 1):
            loss = train_one_epoch(model, train_loader, optimizer, criterion)
            epsilon = (
                float(privacy_engine.get_epsilon(TARGET_DELTA))
                if formal_dp else np.nan
            )
            history_rows.append({
                "condition": condition,
                "shadow_seed": seed,
                "epoch": epoch,
                "train_loss": loss,
                "epsilon": epsilon,
                "delta": TARGET_DELTA if formal_dp else np.nan,
            })
            if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
                print(
                    f"{condition}, shadow {seed}: "
                    f"epoch {epoch:02d}/{EPOCHS}, loss={loss:.6f}"
                )
    warning_rows.extend(
        summarize_warnings(caught, "shadow_training", condition, seed)
    )
    training_seconds = time.time() - started_at

    actual_epsilon = (
        float(privacy_engine.get_epsilon(TARGET_DELTA)) if formal_dp else None
    )
    if formal_dp:
        assert abs(actual_epsilon - requested_epsilon) <= 0.10

    member_probabilities = predict_probabilities(model, X_members, y_members)
    nonmember_probabilities = predict_probabilities(
        model, X_nonmembers, y_nonmembers
    )
    features = pd.concat([
        create_mia_feature_frame(
            condition, "shadow_member", seed, 1,
            shadow_members, member_probabilities,
        ),
        create_mia_feature_frame(
            condition, "shadow_nonmember", seed, 0,
            shadow_nonmembers, nonmember_probabilities,
        ),
    ], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

    features.to_csv(paths["features"], index=False)
    pd.DataFrame(history_rows).to_csv(paths["history"], index=False)
    pd.DataFrame(
        warning_rows,
        columns=["stage", "condition", "seed", "category", "message", "occurrences"],
    ).to_csv(paths["warnings"], index=False)

    config = {
        "cache_signature": signature,
        "condition": condition,
        "shadow_seed": seed,
        "formal_dp": formal_dp,
        "member_rows": len(shadow_members),
        "nonmember_rows": len(shadow_nonmembers),
        "input_dim": input_dim,
        "target_epsilon": requested_epsilon,
        "actual_epsilon": actual_epsilon,
        "absolute_epsilon_error": (
            abs(actual_epsilon - requested_epsilon) if formal_dp else None
        ),
        "delta": TARGET_DELTA if formal_dp else None,
        "noise_multiplier": noise_multiplier,
        "max_grad_norm": MAX_GRAD_NORM if formal_dp else None,
        "sample_rate": sample_rate,
        "training_seconds": training_seconds,
        "features_sha256": calculate_sha256(paths["features"]),
        "history_sha256": calculate_sha256(paths["history"]),
        "preprocessing_fit_partition": "this shadow model's member half only",
    }
    write_json(paths["config"], config)

    del model, optimizer, train_loader, preprocessor
    del X_members, X_nonmembers, y_members, y_nonmembers
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return features, config, warning_rows

shadow_frames_by_condition = {}
shadow_config_rows = []
all_warning_rows = []

for spec in CONDITIONS:
    condition_frames = []
    for seed in SHADOW_SEEDS:
        features, config, warning_rows = train_shadow(spec, seed)
        condition_frames.append(features)
        shadow_config_rows.append(config)
        all_warning_rows.extend(warning_rows)
    shadow_frames_by_condition[spec["condition"]] = pd.concat(
        condition_frames, ignore_index=True
    )

shadow_configs = pd.DataFrame(shadow_config_rows)
dp_shadows = shadow_configs[shadow_configs["formal_dp"] == True]
SHADOW_BUDGET_GATE = bool(
    (dp_shadows["absolute_epsilon_error"].astype(float) <= 0.10).all()
    and np.allclose(dp_shadows["delta"].astype(float), TARGET_DELTA)
)
assert SHADOW_BUDGET_GATE
print("All 15 condition-matched shadows complete; budget gate passed.")


## 11. Calibrate the two fixed attacks from shadow outputs only


In [ ]:
def calculate_mia_advantage(labels: np.ndarray, scores: np.ndarray) -> float:
    fpr, tpr, _ = roc_curve(labels, scores)
    return float(np.max(tpr - fpr))

def calculate_tpr_at_fpr(
    labels: np.ndarray,
    scores: np.ndarray,
    maximum_fpr: float,
) -> float:
    fpr, tpr, _ = roc_curve(labels, scores)
    eligible = fpr <= maximum_fpr
    return float(np.max(tpr[eligible])) if eligible.any() else 0.0

def find_balanced_accuracy_threshold(labels: np.ndarray, scores: np.ndarray):
    candidates = np.unique(np.quantile(scores, np.linspace(0.001, 0.999, 400)))
    best_threshold = float(candidates[0])
    best_score = -1.0
    for threshold in candidates:
        score = balanced_accuracy_score(labels, scores >= threshold)
        if score > best_score:
            best_threshold = float(threshold)
            best_score = float(score)
    return best_threshold, best_score

@dataclass
class CalibratedAttack:
    condition: str
    threat_model: str
    attack_model: str
    feature_set: str
    operating_threshold: float
    shadow_calibration_auc: float
    score_function: Callable[[pd.DataFrame], np.ndarray]

def calibrate_fixed_attacks(condition: str, shadow_data: pd.DataFrame):
    attack_train = shadow_data[
        shadow_data["seed"].isin(ATTACK_TRAIN_SHADOW_SEEDS)
    ].copy()
    calibration = shadow_data[
        shadow_data["seed"] == CALIBRATION_SHADOW_SEED
    ].copy()
    assert set(attack_train["seed"]).isdisjoint(set(calibration["seed"]))

    score_model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=TARGET_SEED,
        )),
    ])
    score_model.fit(attack_train[["prob_attack"]], attack_train["membership"])
    score_calibration_scores = score_model.predict_proba(
        calibration[["prob_attack"]]
    )[:, 1]
    score_threshold, _ = find_balanced_accuracy_threshold(
        calibration["membership"].to_numpy(), score_calibration_scores
    )

    loss_calibration_scores = -calibration["loss"].to_numpy()
    loss_threshold, _ = find_balanced_accuracy_threshold(
        calibration["membership"].to_numpy(), loss_calibration_scores
    )

    attacks = [
        CalibratedAttack(
            condition,
            "score_only_black_box",
            "logistic_regression",
            "prob_attack",
            score_threshold,
            roc_auc_score(calibration["membership"], score_calibration_scores),
            lambda dataframe, model=score_model: model.predict_proba(
                dataframe[["prob_attack"]]
            )[:, 1],
        ),
        CalibratedAttack(
            condition,
            "label_aware_audit",
            "loss_threshold",
            "loss",
            loss_threshold,
            roc_auc_score(calibration["membership"], loss_calibration_scores),
            lambda dataframe: -dataframe["loss"].to_numpy(),
        ),
    ]

    summary = pd.DataFrame([
        {
            "condition": condition,
            "threat_model": attack.threat_model,
            "attack_model": attack.attack_model,
            "feature_set": attack.feature_set,
            "operating_threshold": attack.operating_threshold,
            "shadow_calibration_auc": attack.shadow_calibration_auc,
            "attacker_training_shadow_seeds": ",".join(
                str(seed) for seed in ATTACK_TRAIN_SHADOW_SEEDS
            ) if attack.attack_model == "logistic_regression" else "none",
            "calibration_shadow_seed": CALIBRATION_SHADOW_SEED,
            "target_outputs_used_for_training_or_calibration": False,
        }
        for attack in attacks
    ])
    return attacks, summary

attacks_by_condition = {}
calibration_frames = []
for condition, shadow_data in shadow_frames_by_condition.items():
    attacks, summary = calibrate_fixed_attacks(condition, shadow_data)
    attacks_by_condition[condition] = attacks
    calibration_frames.append(summary)

attack_calibration = pd.concat(calibration_frames, ignore_index=True)
assert len(attack_calibration) == 6
display(attack_calibration)


## 12. Evaluate MIA and bootstrap uncertainty


In [ ]:
def stratified_bootstrap_indices(labels: np.ndarray, rng: np.random.Generator):
    parts = []
    for value in sorted(np.unique(labels)):
        positions = np.flatnonzero(labels == value)
        parts.append(rng.choice(positions, size=len(positions), replace=True))
    return np.concatenate(parts)

def bootstrap_interval(
    labels: np.ndarray,
    scores: np.ndarray,
    metric: Callable,
    seed: int,
) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(BOOTSTRAP_N):
        indices = stratified_bootstrap_indices(labels, rng)
        values.append(metric(labels[indices], scores[indices]))
    low, high = np.quantile(values, [0.025, 0.975])
    return float(low), float(high)

def evaluate_attack(
    attack: CalibratedAttack,
    data: pd.DataFrame,
    subset: str,
    with_ci: bool,
) -> dict:
    labels = data["membership"].to_numpy(dtype=int)
    scores = attack.score_function(data)
    predictions = (scores >= attack.operating_threshold).astype(int)
    auc = roc_auc_score(labels, scores)
    advantage = calculate_mia_advantage(labels, scores)
    if with_ci:
        auc_low, auc_high = bootstrap_interval(
            labels, scores, roc_auc_score,
            stable_seed(attack.condition, attack.threat_model, subset, "auc"),
        )
        adv_low, adv_high = bootstrap_interval(
            labels, scores, calculate_mia_advantage,
            stable_seed(attack.condition, attack.threat_model, subset, "advantage"),
        )
    else:
        auc_low = auc_high = adv_low = adv_high = np.nan
    return {
        "condition": attack.condition,
        "subset": subset,
        "threat_model": attack.threat_model,
        "attack_model": attack.attack_model,
        "feature_set": attack.feature_set,
        "n": len(data),
        "members": int(labels.sum()),
        "nonmembers": int((labels == 0).sum()),
        "mia_auc": auc,
        "mia_auc_ci_low": auc_low,
        "mia_auc_ci_high": auc_high,
        "mia_advantage": advantage,
        "mia_advantage_ci_low": adv_low,
        "mia_advantage_ci_high": adv_high,
        "mia_balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "mia_precision": precision_score(labels, predictions, zero_division=0),
        "mia_recall": recall_score(labels, predictions, zero_division=0),
        "tpr_at_1pct_fpr": calculate_tpr_at_fpr(labels, scores, 0.01),
        "tpr_at_5pct_fpr": calculate_tpr_at_fpr(labels, scores, 0.05),
        "operating_threshold": attack.operating_threshold,
        "shadow_calibration_auc": attack.shadow_calibration_auc,
        "bootstrap_n": BOOTSTRAP_N if with_ci else 0,
    }

overall_rows = []
family_rows = []
for condition, attacks in attacks_by_condition.items():
    target_data = target_results[condition].mia_features
    for attack in attacks:
        overall_rows.append(evaluate_attack(attack, target_data, "overall", True))
        for family in sorted(target_data["attack_family"].unique()):
            subset = target_data[target_data["attack_family"] == family]
            counts = subset["membership"].value_counts()
            if (
                counts.get(1, 0) >= MIN_FAMILY_MEMBERS
                and counts.get(0, 0) >= MIN_FAMILY_NONMEMBERS
            ):
                family_rows.append(evaluate_attack(
                    attack, subset, f"attack_family={family}", False
                ))

mia_results = pd.DataFrame(overall_rows)
mia_by_family = pd.DataFrame(family_rows)
assert len(mia_results) == 6
display(mia_results[[
    "condition", "threat_model", "attack_model", "mia_auc",
    "mia_auc_ci_low", "mia_auc_ci_high", "mia_advantage",
]])


## 13. Paired MIA differences and descriptive IDS differences


In [ ]:
def paired_bootstrap_difference(
    labels: np.ndarray,
    reference_scores: np.ndarray,
    comparison_scores: np.ndarray,
    metric: Callable,
    seed: int,
) -> tuple[float, float, float]:
    estimate = float(
        metric(labels, comparison_scores) - metric(labels, reference_scores)
    )
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(BOOTSTRAP_N):
        indices = stratified_bootstrap_indices(labels, rng)
        values.append(
            metric(labels[indices], comparison_scores[indices])
            - metric(labels[indices], reference_scores[indices])
        )
    low, high = np.quantile(values, [0.025, 0.975])
    return estimate, float(low), float(high)

reference_data = target_results["non_private"].mia_features.sort_values(
    ["membership", "record_id"]
).reset_index(drop=True)
paired_rows = []
for dp_condition in ["dp_eps_4", "dp_eps_2"]:
    comparison_data = target_results[dp_condition].mia_features.sort_values(
        ["membership", "record_id"]
    ).reset_index(drop=True)
    pd.testing.assert_frame_equal(
        reference_data[["membership", "record_id"]],
        comparison_data[["membership", "record_id"]],
        check_dtype=False,
    )
    labels = reference_data["membership"].to_numpy(dtype=int)
    for threat_model in ["score_only_black_box", "label_aware_audit"]:
        reference_attack = next(
            attack for attack in attacks_by_condition["non_private"]
            if attack.threat_model == threat_model
        )
        comparison_attack = next(
            attack for attack in attacks_by_condition[dp_condition]
            if attack.threat_model == threat_model
        )
        reference_scores = reference_attack.score_function(reference_data)
        comparison_scores = comparison_attack.score_function(comparison_data)
        for metric_name, metric in [
            ("mia_auc", roc_auc_score),
            ("mia_advantage", calculate_mia_advantage),
        ]:
            estimate, low, high = paired_bootstrap_difference(
                labels, reference_scores, comparison_scores, metric,
                stable_seed(dp_condition, threat_model, metric_name, "paired"),
            )
            paired_rows.append({
                "reference_condition": "non_private",
                "comparison_condition": dp_condition,
                "threat_model": threat_model,
                "reference_attack_model": reference_attack.attack_model,
                "comparison_attack_model": comparison_attack.attack_model,
                "metric": metric_name,
                "n": len(labels),
                "reference_estimate": metric(labels, reference_scores),
                "comparison_estimate": metric(labels, comparison_scores),
                "difference_dp_minus_non_private": estimate,
                "difference_ci_low": low,
                "difference_ci_high": high,
                "supports_measured_reduction": bool(high < 0),
                "bootstrap_n": BOOTSTRAP_N,
            })

paired_mia_differences = pd.DataFrame(paired_rows)

ids_results = pd.concat(
    [result.ids for result in target_results.values()],
    ignore_index=True,
)
final_ids = ids_results[
    (ids_results["split"] == "official_UNSW_NB15_test")
    & (ids_results["threshold_policy"] == "validation_selected_F2")
].set_index("condition")

utility_difference_rows = []
for dp_condition in ["dp_eps_4", "dp_eps_2"]:
    for metric in ["recall", "fnr", "f1", "fpr", "pr_auc"]:
        utility_difference_rows.append({
            "reference_condition": "non_private",
            "comparison_condition": dp_condition,
            "metric": metric,
            "reference_value": final_ids.loc["non_private", metric],
            "comparison_value": final_ids.loc[dp_condition, metric],
            "difference_dp_minus_non_private": (
                final_ids.loc[dp_condition, metric]
                - final_ids.loc["non_private", metric]
            ),
            "uncertainty": "not estimated; single target-training seed",
        })
utility_differences = pd.DataFrame(utility_difference_rows)

display(final_ids[["actual_epsilon", "threshold", "recall", "fnr", "f1", "fpr", "pr_auc"]])
display(paired_mia_differences)


## 14. Publication-oriented figures


In [ ]:
FIGURE_DIR = RESULTS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
condition_order = ["non_private", "dp_eps_4", "dp_eps_2"]
condition_labels = ["Non-private", "DP ε≈4", "DP ε≈2"]
colors = ["#4C78A8", "#F58518", "#E45756"]

utility_plot = final_ids.loc[condition_order]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for axis, metric, title in zip(
    axes,
    ["recall", "f1", "fpr", "pr_auc"],
    ["Recall", "F1", "False-positive rate", "Average precision"],
):
    axis.bar(condition_labels, utility_plot[metric], color=colors)
    axis.set_title(title)
    axis.set_ylim(0, max(1.0, float(utility_plot[metric].max()) * 1.08))
    axis.tick_params(axis="x", rotation=25)
fig.suptitle("UNSW-NB15 single-seed external IDS utility")
fig.tight_layout()
for suffix in ["png", "pdf"]:
    fig.savefig(FIGURE_DIR / f"01_unsw_ids_utility.{suffix}", dpi=220, bbox_inches="tight")
plt.show()

fig, axis = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(condition_order))
width = 0.34
for offset, threat_model, label in [
    (-width / 2, "score_only_black_box", "Score-only LR"),
    (width / 2, "label_aware_audit", "Label-aware loss threshold"),
]:
    subset = mia_results[
        mia_results["threat_model"] == threat_model
    ].set_index("condition").loc[condition_order]
    y = subset["mia_auc"].to_numpy()
    lower = y - subset["mia_auc_ci_low"].to_numpy()
    upper = subset["mia_auc_ci_high"].to_numpy() - y
    axis.errorbar(
        x + offset, y, yerr=np.vstack([lower, upper]), fmt="o",
        capsize=4, label=label,
    )
axis.axhline(0.5, color="black", linestyle="--", linewidth=1, label="Chance")
axis.set_xticks(x, condition_labels)
axis.set_ylabel("MIA ROC-AUC")
axis.set_title("UNSW-NB15 MIA AUC with 95% stratified bootstrap intervals")
axis.legend()
fig.tight_layout()
for suffix in ["png", "pdf"]:
    fig.savefig(FIGURE_DIR / f"02_unsw_mia_auc.{suffix}", dpi=220, bbox_inches="tight")
plt.show()

score_mia = mia_results[
    mia_results["threat_model"] == "score_only_black_box"
].set_index("condition").loc[condition_order]
fig, axis = plt.subplots(figsize=(6.5, 4.8))
for condition, label, color in zip(condition_order, condition_labels, colors):
    axis.scatter(
        score_mia.loc[condition, "mia_auc"],
        utility_plot.loc[condition, "f1"],
        s=100, color=color, label=label,
    )
axis.axvline(0.5, color="black", linestyle="--", linewidth=1)
axis.set_xlabel("Score-only MIA ROC-AUC")
axis.set_ylabel("IDS F1 (validation-selected threshold)")
axis.set_title("UNSW-NB15 descriptive privacy–utility view")
axis.legend()
fig.tight_layout()
for suffix in ["png", "pdf"]:
    fig.savefig(FIGURE_DIR / f"03_unsw_privacy_utility.{suffix}", dpi=220, bbox_inches="tight")
plt.show()


## 15. Save complete evidence, checksums, and final gates


In [ ]:
target_configs = pd.DataFrame([
    {
        **load_json(target_cache_paths(condition)["config"]),
        "model_path": str(result.model_path),
    }
    for condition, result in target_results.items()
])
target_histories = pd.concat(
    [result.history for result in target_results.values()], ignore_index=True
)
threshold_searches = pd.concat([
    result.threshold_search.assign(condition=condition)
    for condition, result in target_results.items()
], ignore_index=True)
target_mia_features = pd.concat(
    [result.mia_features for result in target_results.values()], ignore_index=True
)
shadow_mia_features = pd.concat(
    list(shadow_frames_by_condition.values()), ignore_index=True
)

combined_warning_rows = [
    *all_warning_rows,
    *[
        warning
        for result in target_results.values()
        for warning in result.warnings
    ],
]
warning_table = pd.DataFrame(
    combined_warning_rows,
    columns=["stage", "condition", "seed", "category", "message", "occurrences"],
)
if not warning_table.empty:
    warning_table = warning_table.groupby(
        ["stage", "condition", "seed", "category", "message"],
        dropna=False,
        as_index=False,
    )["occurrences"].sum()

table_outputs = {
    "unsw_nb15_ids_results.csv": ids_results,
    "unsw_nb15_target_configs.csv": target_configs,
    "unsw_nb15_target_training_history.csv": target_histories,
    "unsw_nb15_threshold_search.csv": threshold_searches,
    "unsw_nb15_shadow_configs.csv": shadow_configs,
    "unsw_nb15_attack_calibration.csv": attack_calibration,
    "unsw_nb15_mia_results.csv": mia_results,
    "unsw_nb15_mia_by_attack_family.csv": mia_by_family,
    "unsw_nb15_paired_mia_differences.csv": paired_mia_differences,
    "unsw_nb15_descriptive_utility_differences.csv": utility_differences,
    "unsw_nb15_target_mia_features.csv": target_mia_features,
    "unsw_nb15_shadow_mia_features.csv": shadow_mia_features,
    "opacus_warning_summary.csv": warning_table,
}
for filename, dataframe in table_outputs.items():
    dataframe.to_csv(RESULTS_DIR / filename, index=False)

experiment_config = {
    **protocol_core,
    "protocol_fingerprint": PROTOCOL_FINGERPRINT,
    "experiment": "09_unsw_nb15_external_validation",
    "status_scope": "supplementary single-seed external validation",
    "target_delta": TARGET_DELTA,
    "split": "70% target_train / 10% target_validation / 20% shadow_pool",
    "official_test_use": "final IDS utility only",
    "threshold_policy": "F2 selected on target_validation only",
    "preprocessing": preprocessing_manifest,
    "mia": {
        "attacks": FIXED_ATTACKS,
        "attacker_training_shadow_seeds": ATTACK_TRAIN_SHADOW_SEEDS,
        "calibration_shadow_seed": CALIBRATION_SHADOW_SEED,
        "target_members": "balanced sample from target_train",
        "target_nonmembers": "balanced sample from target_validation",
        "bootstrap_n": BOOTSTRAP_N,
        "bootstrap_scope": (
            "record-sampling uncertainty conditional on one trained target model; "
            "not target-training-seed uncertainty"
        ),
        "family_diagnostics": "descriptive only; no bootstrap intervals",
    },
    "reporting_boundaries": [
        "Do not call epsilon 4 optimal.",
        "Do not claim generalisation from one target-training seed.",
        "Do not claim DP reduced measured leakage unless the paired interval is below zero.",
        "Formal DP applies to optimisation conditional on fixed preprocessing.",
        "UNSW-NB15 results are supplementary external evidence.",
    ],
}
write_json(RESULTS_DIR / "unsw_nb15_config.json", experiment_config)

target_condition_gate = set(target_results) == {
    "non_private", "dp_eps_4", "dp_eps_2"
}
epsilon_gate = all(
    result.actual_epsilon is not None
    and 0 < result.actual_epsilon <= result.target_epsilon + 0.10
    for result in target_results.values() if result.formal_dp
)
ids_gate = bool(
    len(ids_results) == 9
    and set(ids_results["condition"]) == set(target_results)
    and len(final_ids) == 3
)
fixed_attack_gate = bool(
    len(mia_results) == 6
    and set(zip(mia_results["threat_model"], mia_results["attack_model"]))
    == {
        ("score_only_black_box", "logistic_regression"),
        ("label_aware_audit", "loss_threshold"),
    }
)
shadow_gate = bool(
    len(shadow_configs) == 15
    and set(shadow_configs["shadow_seed"]) == set(SHADOW_SEEDS)
    and shadow_configs.groupby("condition")["shadow_seed"].nunique().eq(5).all()
    and SHADOW_BUDGET_GATE
)
split_gate = bool(
    split_manifest.set_index("partition").loc["target_train", "rows"]
    + split_manifest.set_index("partition").loc["target_validation", "rows"]
    + split_manifest.set_index("partition").loc["shadow_pool", "rows"]
    == len(train_df)
)
no_test_tuning_gate = bool(
    all(
        set(result.mia_features["partition"]) == {
            "target_train", "target_validation"
        }
        for result in target_results.values()
    )
    and set(target_mia_sample_manifest["partition"]) == {
        "target_train", "target_validation"
    }
)
evidence_values_gate = bool(
    ids_results[["recall", "fnr", "f1", "fpr", "pr_auc"]].notna().all().all()
    and mia_results[["mia_auc", "mia_advantage"]].notna().all().all()
)

gates = {
    "dataset_schema_and_rows": True,
    "disjoint_complete_development_split": split_gate,
    "target_preprocessing_fit_only_on_target_train": True,
    "three_pre_specified_target_conditions": target_condition_gate,
    "private_target_epsilon_accounting": epsilon_gate,
    "fifteen_condition_matched_shadows": shadow_gate,
    "two_fixed_attack_families_only": fixed_attack_gate,
    "official_test_excluded_from_tuning_and_mia": no_test_tuning_gate,
    "ids_output_coverage": ids_gate,
    "finite_primary_evidence": evidence_values_gate,
}
COMPLETE = bool(all(gates.values()))
assert COMPLETE, {key: value for key, value in gates.items() if not value}

interpretation_lines = [
    "# Experiment 09 interpretation boundary",
    "",
    "This file is generated from a single target-training seed on UNSW-NB15.",
    "It is supplementary external evidence, not a new optimisation study.",
    "",
    "## Final IDS utility (validation-selected threshold)",
    "",
    "```text",
    final_ids.reset_index()[[
        "condition", "actual_epsilon", "threshold", "recall", "fnr", "f1", "fpr", "pr_auc"
    ]].to_string(index=False),
    "```",
    "",
    "## Overall membership-inference results",
    "",
    "```text",
    mia_results[[
        "condition", "threat_model", "attack_model", "mia_auc",
        "mia_auc_ci_low", "mia_auc_ci_high", "mia_advantage"
    ]].to_string(index=False),
    "```",
    "",
    "## Required claim boundaries",
    "",
    "- Do not call epsilon 4 optimal.",
    "- Do not generalise stability from one target-training seed.",
    "- Do not claim measured leakage reduction unless a paired DP-minus-non-private interval is entirely below zero.",
    "- Keep formal DP distinct from empirical MIA performance.",
    "- State that DP applies to optimisation conditional on fixed preprocessing.",
]
interpretation_path = RESULTS_DIR / "interpretation.md"
interpretation_path.write_text("\n".join(interpretation_lines), encoding="utf-8")

run_status = {
    "experiment": "09_unsw_nb15_external_validation",
    "status": "COMPLETE",
    "protocol_version": PROTOCOL_VERSION,
    "protocol_fingerprint": PROTOCOL_FINGERPRINT,
    "completed_at_unix": time.time(),
    "device": str(DEVICE),
    "trainings": {"target": 3, "shadow": 15, "total": 18},
    "gates": gates,
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "torch": torch.__version__,
        "opacus": __import__("opacus").__version__,
    },
}
write_json(RESULTS_DIR / "run_status.json", run_status)

provenance_dir = RESULTS_DIR / "provenance"
provenance_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(split_indices_file, provenance_dir / split_indices_file.name)
shutil.copy2(preprocessor_file, provenance_dir / preprocessor_file.name)

expected_result_names = [
    "unsw_nb15_dataset_identity.json",
    "unsw_nb15_split_manifest.csv",
    "unsw_nb15_preprocessing_manifest.json",
    "unsw_nb15_config.json",
    "run_status.json",
    "interpretation.md",
    *table_outputs.keys(),
]
evidence_candidates = [RESULTS_DIR / name for name in expected_result_names]
evidence_candidates += sorted(FIGURE_DIR.glob("01_unsw_ids_utility.*"))
evidence_candidates += sorted(FIGURE_DIR.glob("02_unsw_mia_auc.*"))
evidence_candidates += sorted(FIGURE_DIR.glob("03_unsw_privacy_utility.*"))
evidence_candidates += [
    provenance_dir / split_indices_file.name,
    provenance_dir / preprocessor_file.name,
]
missing_evidence = [str(path) for path in evidence_candidates if not path.exists()]
assert not missing_evidence, {"missing_evidence": missing_evidence}
evidence_hashes = {
    str(path.relative_to(RESULTS_DIR)): {
        "sha256": calculate_sha256(path),
        "bytes": path.stat().st_size,
    }
    for path in sorted(evidence_candidates)
}
evidence_manifest = {
    "experiment": "09_unsw_nb15_external_validation",
    "status": "COMPLETE",
    "protocol_version": PROTOCOL_VERSION,
    "protocol_fingerprint": PROTOCOL_FINGERPRINT,
    "dataset_hashes": {"train": TRAIN_SHA256, "test": TEST_SHA256},
    "gates": gates,
    "file_count_excluding_manifest": len(evidence_hashes),
    "files": evidence_hashes,
    "raw_dataset_included": False,
    "model_state_files_included": False,
}
evidence_manifest_path = RESULTS_DIR / "evidence_manifest.json"
write_json(evidence_manifest_path, evidence_manifest)

evidence_zip = RESULTS_DIR / "experiment09_evidence.zip"
with zipfile.ZipFile(evidence_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(evidence_candidates + [evidence_manifest_path]):
        archive.write(
            path,
            Path("experiment09_evidence") / path.relative_to(RESULTS_DIR),
        )

print(json.dumps(gates, indent=2))
print("Evidence ZIP:", evidence_zip)
print("Evidence ZIP SHA-256:", calculate_sha256(evidence_zip))
print("\nEXPERIMENT 09: COMPLETE")


## 16. Stop and review

Do not begin paper drafting from a partial run. Share both of these after the final cell
reports `EXPERIMENT 09: COMPLETE`:

1. `experiment09_evidence.zip`
2. this executed notebook, saved from Colab

The evidence must be audited before deciding whether UNSW-NB15 supports, weakens, or
contradicts the NSL-KDD findings. No universal-performance or optimal-ε claim is allowed.
